# ArNetECG - Raw ECG Preprocessing and Model Prediction

This notebook demonstrates the workflow for:
1. **Preprocessing Raw ECG data** from the **SHDB-AF** dataset for prediction.
2. **Running the trained ArNetECG model** for **AF prediction** using the preprocessed data.


## Key differences from RR prediction:
- **Input**: Raw ECG windows instead of RR intervals
- **Model**: ArNetECG (uses ResNetECG feature extractor + temporal model)
- **Output**: One prediction per 30-second window
---

### Dataset:
- **SHDB-AF**: The dataset used in this example is the [SHDB-AF dataset](https://physionet.org/content/shdb-af/1.0.1/), which contains ECG signals annotated with peak information and AF-related labels.

### Dataset Details:
- **ECG signals**: The data consists of ECG recordings, where each sample is labeled with corresponding **R-peaks annotations**.
- **R-peak annotations**: The location of R-peaks in the ECG signal.
- **AF labels per peak**: Each R-peak is annotated with a label indicating whether it is associated with **AF** or not.
- **Overall patient label**: The dataset includes a **global label** for each patient indicating the overall AF status (e.g., **PAF**: Paroxysmal AF, **Per**: Persistent AF, **Non-AF**).

This notebook will help demonstrate how to prepare the data for prediction and how to use the trained **ArNetECG model** to make predictions for **AF detection**.


### 1. Import Libraries

In [1]:
import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)

import yaml
import numpy as np
import pandas as pd
import wfdb
import subprocess
from tqdm.notebook import tqdm  # Import tqdm for Jupyter Notebooks
import nb_utils
from pathlib import Path

### 2. Setting Up the Paths
We will first set up paths for data storage and where to save the processed data:

In [2]:
# Setup directories
db_path = '.././physionet_data'  # Where you'll download the PhysioNet dataset
input_path = '.././data/ecg_windows'  # Where to save the processed data which will be used as input to the model
results_dir = '../results'  # Where to save the predictions
physiozoo_dir = results_dir + '/physiozoo_rhythms'
predictions_dir = results_dir + '/predictions'
statistics_dir = results_dir + '/af_statistics'
model_dir = '../ArNetECG_exported_model'  # Where the .pb model is saved

# Create directories if they don't exist
os.makedirs(db_path, exist_ok=True)
os.makedirs(input_path, exist_ok=True)
os.makedirs(physiozoo_dir, exist_ok=True)
os.makedirs(predictions_dir, exist_ok=True)
os.makedirs(statistics_dir, exist_ok=True)


## 3. Setting up parameters
Constant model parameters

In [3]:
def load_ecg_record(record_name, db_path='../physionet_data'):
    """Load ECG record and return signal amplitudes and timestamps"""
    try:
        record = wfdb.rdrecord(f'{db_path}/{record_name}')

        # Check sampling rate
        if record.fs != SAMPLING_RATE:
            print(f'Warning: {record_name} has fs={record.fs}, expected {SAMPLING_RATE}')
            return None, None

        # Get ECG signal for specified lead (convert to 0-based indexing)
        signal = record.p_signal[:, LEAD-1]

        # Calculate timestamps: sample_index / sampling_rate
        timestamps = np.arange(len(signal)) / SAMPLING_RATE

        print(f'Loaded {record_name}: {len(signal)} samples, fs={record.fs}Hz, duration={timestamps[-1]:.1f}s')
        return signal, timestamps

    except Exception as e:
        print(f'Error loading {record_name}: {e}')
        return None, None


def create_ecg_windows(signal):
    """Create 30-second windows from continuous ECG signal"""
    n_windows = len(signal) // WINDOW_SIZE_SAMPLES
    windows = []

    for i in range(n_windows):
        start = i * WINDOW_SIZE_SAMPLES
        end = (i + 1) * WINDOW_SIZE_SAMPLES
        window = signal[start:end]
        windows.append(window)

    return np.array(windows)


def format_ecg_data_for_arnetecg(ecg_windows, record_id):
    """
    Format ECG windows for ArNetECG prediction.
    
    ArNetECG expects input format: [ECG_data (6000), prec_windows, succ_windows, global_label, ids]
    Column positions: [0:6000, -4, -3, -2, -1]

    Parameters:
    -----------
    ecg_windows : np.ndarray
        Array of shape (n_windows, 6000) containing ECG windows
    record_id : str
        Patient/record identifier

    Returns:
    --------
    np.ndarray
        Formatted data with shape (n_windows, 6005) where columns are:
        [ECG_data (6000), prec_windows, succ_windows, global_label, ids]
    """
    n_windows = len(ecg_windows)

    # Ensure ECG data is float32
    ecg_data = ecg_windows.astype(np.float32)

    # prec_windows: number of preceding windows available (index for sequential windows)
    prec_windows = np.arange(n_windows, dtype=np.float32)

    # succ_windows: number of succeeding windows available
    succ_windows = (np.arange(n_windows, 0, -1, dtype=np.float32) - 1)

    # global_label: set to 1 for all windows (doesn't affect prediction)
    global_label = np.ones(n_windows, dtype=np.float32)

    # ids: patient/record identifier - convert to numeric for storage, will be converted back to string in model
    # Use a simple numeric encoding: hash the record_id or use a numeric ID
    # For simplicity, if record_id is numeric, use it directly; otherwise use hash
    try:
        numeric_id = float(record_id)
    except (ValueError, TypeError):
        # Use hash if not numeric (modulo to keep it reasonable)
        numeric_id = float(hash(str(record_id)) % 1000000)

    ids = np.full(n_windows, numeric_id, dtype=np.float32)

    # Combine into the expected format: [ECG_data (6000), prec_windows, succ_windows, global_label, ids]
    # All as float32 to avoid dtype=object issues with TensorFlow
    formatted_data = np.concatenate([
        ecg_data,  # Shape: (n_windows, 6000)
        prec_windows.reshape(-1, 1),  # Shape: (n_windows, 1)
        succ_windows.reshape(-1, 1),  # Shape: (n_windows, 1)
        global_label.reshape(-1, 1),  # Shape: (n_windows, 1)
        ids.reshape(-1, 1)  # Shape: (n_windows, 1)
    ], axis=1)

    return formatted_data


In [4]:
# ECG parameters
WINDOW_SIZE_SEC = 30
SAMPLING_RATE = 200
WINDOW_SIZE_SAMPLES = WINDOW_SIZE_SEC * SAMPLING_RATE  # 6000
LEAD = 1

### 4. Download the Data
Next, we will download the required ECG signal data and annotation files for SHDB-AF:

In [5]:
# Download annotation files and additional data
nb_utils.download_file(f'https://physionet.org/files/shdb-af/1.0.1/AdditionalData.csv', f'{db_path}/AdditionalData.csv')

additionaldata = pd.read_csv(f'{db_path}/AdditionalData.csv')
record_names = additionaldata.Data_ID.astype(str).str.zfill(3)

In [6]:
for record_name in tqdm(record_names, desc="Downloading ECG records", leave=False):
    filepath = f'{db_path}/{record_name}'
    url = f'https://physionet.org/files/shdb-af/1.0.1/{record_name}'
    #  download records and annotations
    if nb_utils.url_exists(f"{url}.atr") and not os.path.exists(f"{filepath}.hea"):
        nb_utils.download_file(f'{url}.atr', f'{filepath}.atr')
        nb_utils.download_file(f'{url}.qrs', f'{filepath}.qrs')
        nb_utils.download_file(f'{url}.hea', f'{filepath}.hea')
        nb_utils.download_file(f'{url}.dat', f'{filepath}.dat')

### 5. Preprocess annotation Data

We will preprocess the raw ECG, time stamps and associated recording ids

In [7]:
all_y_df = []  # list to collect per-record true labels Dataframes
for record_name in tqdm(record_names, desc="Processing ECG records", leave=False):
    filepath = f"{db_path}/{record_name}"

    try:
        if not os.path.exists(f"{filepath}.atr"):
            continue
        # Load annotations (e.g., R-peaks)

        else:
                # Load record
            signal, timestamps = load_ecg_record(record_name, db_path=db_path)


            # Create ECG windows
            ecg_windows = create_ecg_windows(signal)
            # print(f"Created {len(ecg_windows)} ECG windows")
            # print(f"Each window: {WINDOW_SIZE_SAMPLES} samples ({WINDOW_SIZE_SEC}s @ {SAMPLING_RATE}Hz)")
            # print(f"Shape: {ecg_windows.shape}")

            # Format data for ArNetECG (add prec_windows, succ_windows, global_label, ids)
            formatted_data = format_ecg_data_for_arnetecg(ecg_windows, record_name)

            # Extract R-peak sample indices
            annotation = wfdb.rdann(filepath, 'atr')
            r_peaks = annotation.sample
            rr_time = r_peaks / annotation.fs

            # Extract ecg-based windows
            recording_duration = len(signal) / SAMPLING_RATE
            window_starts, window_ends = nb_utils.create_window_times(recording_duration, WINDOW_SIZE_SEC)
            ecg_windows = nb_utils.create_ecg_windows(signal, window_size_samples=WINDOW_SIZE_SAMPLES)

            # Skip files with too few beats
            if len(ecg_windows) < 2:
                print(f"Skipping {record_name}: recording too short, number of ECG windows: ({len(ecg_windows)})")
                continue

            #  derive y labels
            y = nb_utils.calc_window_labels(rhythm_sequence=annotation.aux_note, window_size=WINDOW_SIZE_SEC, data_type='ecg',
                                            beat_timestamps=rr_time,
                                            window_starts=window_starts, window_ends=window_ends, threshold=0.5)
            y_df = pd.DataFrame({
                'true_label': y,
                'prec_window': np.arange(y.size, dtype=int),
                'patient_id': record_name,
                'start_time': window_starts,
                'end_time': window_ends,
            })
            
            # Save processed data\n
            input_file_pat = f'{input_path}/shdb_id_{record_name}_ecg_windows.npy'
            np.save(input_file_pat, formatted_data)

            # Append to the lists
            all_y_df.append(y_df)
            
    except Exception as e:
        print(f"Error processing {record_name}: {e}")
        continue
        
# Concatenate all DataFrames into one
combined_y_df = pd.concat(all_y_df, ignore_index=True)
print(f"Processed overall {combined_y_df.patient_id.nunique()} patients ..")

Processing ECG records:   0%|          | 0/128 [00:00<?, ?it/s]

Loaded 001: 17220000 samples, fs=200Hz, duration=86100.0s
Loaded 002: 17280000 samples, fs=200Hz, duration=86400.0s
Loaded 003: 17280000 samples, fs=200Hz, duration=86400.0s
Loaded 004: 17340000 samples, fs=200Hz, duration=86700.0s
Loaded 005: 17340000 samples, fs=200Hz, duration=86700.0s
Loaded 006: 17280000 samples, fs=200Hz, duration=86400.0s
Loaded 007: 17306000 samples, fs=200Hz, duration=86530.0s
Loaded 008: 17280000 samples, fs=200Hz, duration=86400.0s
Loaded 009: 17280000 samples, fs=200Hz, duration=86400.0s
Loaded 010: 17280000 samples, fs=200Hz, duration=86400.0s
Loaded 011: 17340000 samples, fs=200Hz, duration=86700.0s
Loaded 012: 17220000 samples, fs=200Hz, duration=86100.0s
Loaded 013: 17220000 samples, fs=200Hz, duration=86100.0s
Loaded 014: 17280000 samples, fs=200Hz, duration=86400.0s
Loaded 015: 17280000 samples, fs=200Hz, duration=86400.0s
Loaded 017: 17340000 samples, fs=200Hz, duration=86700.0s
Loaded 018: 17280000 samples, fs=200Hz, duration=86400.0s
Loaded 019: 17

In [8]:
arrays = []
for record_name in record_names:
    file_path = f"{input_path}/shdb_id_{record_name}_ecg_windows.npy"
    if Path(file_path).exists():
        arrays.append(np.load(file_path))
    else:
        print(f"Missing: {file_path}")

# concat along first axis (windows)
all_data = np.concatenate(arrays, axis=0)

# save combined file
input_file = f"{input_path}/shdb_all_ecg_windows.npy"
np.save(input_file, all_data)

print("Final shape:", all_data.shape)

Missing: .././data/ecg_windows/shdb_id_053_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_057_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_058_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_059_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_060_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_061_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_063_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_066_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_067_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_068_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_069_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_079_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_080_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_081_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_083_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_085_ecg_windows.npy
Missing: .././data/ecg_windows/shdb_id_087_ecg_windows.n

### 6. Run Prediction with the Trained Model (Full Architecture (Parts 1 & 2))
Finally, we use the trained ArNetECG model to make predictions based on the preprocessed data:

In [9]:
subprocess.run(['python', '../run_detection.py',
                '--model_type', 'ArNetECG',
                '--saved_model', model_dir, 
                '--inference_mode', 'full',
                '--input_file', input_file,
                '--output_name', 'shdb_pred_df_full_ArNetECG',
                '--save_output_path', predictions_dir])


2026-02-24 13:54:12.631182: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-24 13:54:12.632730: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-24 13:54:12.662158: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-24 13:54:12.662508: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 13:54:13.074399: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT

Using model: ArNetECG
Predicting...
Results saved to ../results/predictions/shdb_pred_df_full_ArNetECG.csv


CompletedProcess(args=['python', '../run_detection.py', '--model_type', 'ArNetECG', '--saved_model', '../ArNetECG_exported_model', '--inference_mode', 'full', '--input_file', '.././data/ecg_windows/shdb_all_ecg_windows.npy', '--output_name', 'shdb_pred_df_full_ArNetECG', '--save_output_path', '../results/predictions'], returncode=0)

#### 7. Overall model performance

In [10]:
import utils.metrics as metrics
pred_df = pd.read_csv(f"{predictions_dir}/shdb_pred_df_full_ArNetECG.csv")
pred_df.patient_id = pred_df.patient_id.astype(float).astype(int).astype(str).str.zfill(3)

# Merge true label per window to prediction df
combined_pred_df = pred_df.merge(combined_y_df[['patient_id', 'prec_window', 'true_label', 'start_time', 'end_time']], on=['patient_id', 'prec_window'], how='left')


In [11]:
accuracy, fbeta, sensitivity, specificity, PPV, NPV, AUROC, AUCPR = metrics.model_metrics(combined_pred_df.proba, combined_pred_df.true_label, pred_df.pred, print_metrics=True)

Accuracy: 0.9635660245133794
F1-Score: 0.9105544046908465
Sensitivity: 0.9453567848355053
Specificity: 0.9680098212339704
PPV: 0.8782234714438104
NPV: 0.9864113314447592
AUROC: 0.9832867189303827
AUCPR: 0.9066022116734085
[[217627   7192]
 [  2998  51867]]


In [12]:
# Load data for computing mean AFB error
X = np.load(input_file)
m_e_afb = metrics.mean_abs_afb_error(X, np.array(combined_pred_df.pred), np.array(combined_pred_df.true_label), window_size=WINDOW_SIZE_SAMPLES, sampling_rate=SAMPLING_RATE)
print(f"Mean AFb error: {m_e_afb}")

Mean AFb error: 2.96208263334476


#### 7.2 AF event analysis per patient

In [13]:
# Process each patient
rhythm_stats = []
for i, pat in tqdm(enumerate(pred_df.patient_id.unique()), desc="Processing AF statsictics per recording", leave=False):
    name = f'{db_path}/{pat}'
    print(f'Processing recording: {pat}')
    
    # Load record
    record = wfdb.rdrecord(name)
    
    # Save patient predictions
    temp_pred_file = f'{predictions_dir}/pred_{pat}.csv'
    combined_pred_df.loc[combined_pred_df.patient_id.isin([pat])].to_csv(temp_pred_file, index=False)
    
    # Generate PhysioZoo rhythm file
    output_path = f'{physiozoo_dir}/physiozoo_rhythms_{pat}'
    subprocess.run([
        'python', '../../utils/export_arnet2_output_to_physiozoo.py',
        '--input_file', str(temp_pred_file),
        '--output_name', str(output_path)
    ], stdout=subprocess.DEVNULL)
    
    # Read and analyze rhythm events
    pat_df = pd.read_csv(
        f'{str(output_path)}.txt', 
        sep=r"\s+|;|,", 
        skiprows=6, 
        engine='python'
    )
    
    # Compute AF event statistics
    rec_length = len(record.p_signal) / record.fs
    n_events, max_event, min_event, afb = metrics.compute_af_event_statistics(
        rec_length, pat_df
    )
    
    # Store results
    rhythm_stats.append({
        'patient_id': pat,
        'n_events': n_events,
        'max_event_sec': max_event,
        'min_event_sec': min_event,
        'af_burden_pct': 100 * afb
    })

    os.remove(temp_pred_file)

Processing AF statsictics per recording: 0it [00:00, ?it/s]

Processing recording: 001
Number of AF events: 1,
Longest event: 48450.0 sec,
Shortest event: 48450.0 sec,
AF-burden: 56.27%
Processing recording: 010
Number of AF events: 4,
Longest event: 62880.0 sec,
Shortest event: 60.0 sec,
AF-burden: 75.14%
Processing recording: 102
Processing recording: 103
Processing recording: 105
Number of AF events: 3,
Longest event: 4740.0 sec,
Shortest event: 30.0 sec,
AF-burden: 5.56%
Processing recording: 106
Number of AF events: 12,
Longest event: 990.0 sec,
Shortest event: 30.0 sec,
AF-burden: 4.23%
Processing recording: 107
Processing recording: 108
Processing recording: 109
Number of AF events: 8,
Longest event: 4920.0 sec,
Shortest event: 30.0 sec,
AF-burden: 9.62%
Processing recording: 011
Number of AF events: 1,
Longest event: 21210.0 sec,
Shortest event: 21210.0 sec,
AF-burden: 24.46%
Processing recording: 110
Number of AF events: 1,
Longest event: 1050.0 sec,
Shortest event: 1050.0 sec,
AF-burden: 1.22%
Processing recording: 111
Number of AF eve

In [14]:
# Create summary DataFrame
rhythm_df = pd.DataFrame(rhythm_stats)
print("\nAF Event Statistics Summary:")
print(rhythm_df)

# Save summary
summary_path = f"{statistics_dir}/af_event_summary_ArNetECG.csv"
rhythm_df.to_csv(summary_path, index=False)
print(f"\nSummary saved to: {summary_path}")


AF Event Statistics Summary:
   patient_id  n_events  max_event_sec  min_event_sec  af_burden_pct
0         001         1        48450.0        48450.0      56.271777
1         010         4        62880.0           60.0      75.138889
2         102         0            0.0            0.0       0.000000
3         103         0            0.0            0.0       0.000000
4         105         3         4740.0           30.0       5.555556
..        ...       ...            ...            ...            ...
93        077         7         2940.0           30.0       4.062500
94        008         1        59010.0        59010.0      68.298611
95        084         0            0.0            0.0       0.000000
96        086         0            0.0            0.0       0.000000
97        009         1        10470.0        10470.0      12.118056

[98 rows x 5 columns]

Summary saved to: ../results/af_statistics/af_event_summary_ArNetECG.csv
